## 1. Cài đặt môi trường

### 1.1 Import thư viện

In [1]:

# CÀI ĐẶT MÔI TRƯỜNG & ĐỌC DỮ LIỆU

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, r2_score
import math

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị: {device}")

TRAIN_PATH = '/kaggle/input/datasets/traanfddinhfkhair/datasetttt/data/train.csv'
VAL_PATH   = '/kaggle/input/datasets/traanfddinhfkhair/datasetttt/data/val.csv'
TEST_PATH  = '/kaggle/input/datasets/traanfddinhfkhair/datasetttt/data/test.csv'

df_train_raw = pd.read_csv(TRAIN_PATH)
df_val_raw   = pd.read_csv(VAL_PATH)
df_test_raw  = pd.read_csv(TEST_PATH)

df_final_all = pd.concat([df_train_raw, df_val_raw, df_test_raw], ignore_index=True)
df_final_all['TIME'] = pd.to_datetime(df_final_all['TIME'])
df_final_all = df_final_all.sort_values('TIME').reset_index(drop=True)

n_train = len(df_train_raw)
n_val   = len(df_val_raw)
n_test  = len(df_test_raw)

print(f"Train: {n_train:,} | Val: {n_val:,} | Test: {n_test:,} | Total: {len(df_final_all):,}")

Thiết bị: cuda
Train: 82,675 | Val: 10,334 | Test: 10,335 | Total: 103,344


In [2]:

#  ENHANCED FEATURE ENGINEERING - TEMPORAL PATTERN FEATURES (FIXED)

 # Thêm Advanced Temporal Pattern Features
 # Tạo temporal features TRƯỚC khi dropna() để tránh NaN issues

# 1. Weather Forecasting (Dịch 48 bước = 24h)
weather_cols = ['temp', 'rhum', 'prcp', 'wspd']
forecast_weather_features = []
for col in weather_cols:
    new_col = f'{col}_forecast'
    df_final_all[new_col] = df_final_all[col].shift(-48)
    forecast_weather_features.append(new_col)

# 2. Cyclical Encoding
def encode_cyclical(df, col, max_val):
    df[col + '_sin'] = np.sin(2 * np.pi * df[col] / max_val)
    df[col + '_cos'] = np.cos(2 * np.pi * df[col] / max_val)
    return df

df_final_all['Hour']      = df_final_all['TIME'].dt.hour
df_final_all['DayOfWeek'] = df_final_all['TIME'].dt.dayofweek
df_final_all['Month']     = df_final_all['TIME'].dt.month
df_final_all = encode_cyclical(df_final_all, 'Hour', 24)
df_final_all = encode_cyclical(df_final_all, 'DayOfWeek', 7)
df_final_all = encode_cyclical(df_final_all, 'Month', 12)
df_final_all['is_weekend'] = (df_final_all['DayOfWeek'] >= 5).astype(float)
cyclical_cols = ['Hour_sin', 'Hour_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Month_sin', 'Month_cos', 'is_weekend']

# 3. Lag Features
lag_features = []
df_final_all['P224_lag1'] = df_final_all['P_224'].shift(1)
for l in [ 2, 4, 6, 12, 48, 336]:
    col = f'P224_lag{l}'
    df_final_all[col] = df_final_all['P_224'].shift(l)
    lag_features.append(col)

df_final_all['P224_roll24']  = df_final_all['P_224'].shift(1).rolling(24).mean()
df_final_all['P224_roll48']  = df_final_all['P_224'].shift(1).rolling(48).mean()
df_final_all['P224_std24']   = df_final_all['P_224'].shift(1).rolling(24).std()
df_final_all['P224_std48']   = df_final_all['P_224'].shift(1).rolling(48).std()
df_final_all['P224_delta1']  = df_final_all['P_224'].diff(1).shift(1)
df_final_all['P224_delta48'] = df_final_all['P_224'].diff(48).shift(1)
lag_features += ['P224_roll24', 'P224_roll48', 'P224_std24', 'P224_std48', 'P224_delta1', 'P224_delta48']


 # Create Temporal Pattern Features BEFORE dropna()

print(" : Tạo Advanced Temporal Pattern Features...")

def create_temporal_pattern_features_safe(df, target_col='P_224'):
    """ SAFE: Advanced temporal pattern mining với NaN handling"""

    # Intraday patterns (within-day cycles) - SAFE version
    df['intraday_trend'] = df.groupby(df['TIME'].dt.date)[target_col].transform(
        lambda x: x.rolling(6, min_periods=3).mean().diff().fillna(0)  # min_periods + fillna
    )

    # Weekly patterns với lag - REDUCED lag để giảm NaN
    df['weekly_same_hour'] = df[target_col].shift(7*48).fillna(method='bfill')  # Fill NaN
    df['weekly_trend'] = (df[target_col] - df['weekly_same_hour']).fillna(0)

    # Monthly seasonality (sine/cosine for day of month) - NO NaN
    df['monthly_cycle'] = np.sin(2 * np.pi * df['TIME'].dt.day / 30.44)
    df['monthly_phase'] = np.cos(2 * np.pi * df['TIME'].dt.day / 30.44)

    # Load gradient (rate of change) - SAFE version
    df['load_gradient_1h'] = df[target_col].diff(2).fillna(0)  # Fill NaN
    df['load_gradient_3h'] = df[target_col].diff(6).fillna(0)  # Fill NaN
    df['load_acceleration'] = df['load_gradient_1h'].diff(2).fillna(0)  # Fill NaN


    return [
            'intraday_trend',
            'weekly_same_hour', 'weekly_trend',
            # 'monthly_cycle', 'monthly_phase',
            'load_gradient_1h', 'load_gradient_3h', 'load_acceleration'
           ]

#  CREATE TEMPORAL FEATURES BEFORE DROPNA
temporal_pattern_features = create_temporal_pattern_features_safe(df_final_all)
print(f" Đã tạo {len(temporal_pattern_features)} temporal pattern features: {temporal_pattern_features}")


#  FIX LEAKAGE: Xóa bfill(), dùng dropna() và bù trừ n_train

df_final_all = df_final_all.ffill()
len_before = len(df_final_all)
df_final_all = df_final_all.dropna().reset_index(drop=True)
rows_dropped = len_before - len(df_final_all)
n_train = n_train - rows_dropped

print(f"Đã dọn dẹp xong NaN. Cắt bỏ {rows_dropped} dòng mồ côi (chủ yếu ở đầu tập Train).")
assert df_final_all.isna().sum().sum() == 0, "Vẫn còn NaN — kiểm tra lại!"


#  TẠO SẴN DATA NGÀY LỄ (CHƯA GÁN VÀO BASE/TOPO ĐỂ KHÔNG LÀM LỆCH VỊ TRÍ)

VN_HOLIDAY_DATES = [
    "2020-01-01", "2020-01-23", "2020-01-24", "2020-01-25", "2020-01-26", "2020-01-27", "2020-01-28", "2020-01-29",
    "2020-04-02", "2020-04-30", "2020-05-01", "2020-09-02", "2021-01-01", "2021-02-10", "2021-02-11", "2021-02-12",
    "2021-02-13", "2021-02-14", "2021-02-15", "2021-02-16", "2021-04-21", "2021-04-30", "2021-05-01", "2021-09-02",
    "2021-09-03", "2022-01-01", "2022-01-31", "2022-02-01", "2022-02-02", "2022-02-03", "2022-02-04", "2022-04-10",
    "2022-04-11", "2022-04-30", "2022-05-01", "2022-05-02", "2022-05-03", "2022-09-01", "2022-09-02", "2023-01-01",
    "2023-01-02", "2023-01-20", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-01-25", "2023-01-26",
    "2023-04-29", "2023-04-30", "2023-05-01", "2023-05-02", "2023-05-03", "2023-09-01", "2023-09-02", "2023-09-03",
    "2023-09-04", "2024-01-01", "2024-02-08", "2024-02-09", "2024-02-10", "2024-02-11", "2024-02-12", "2024-02-13",
    "2024-02-14", "2024-04-18", "2024-04-30", "2024-05-01", "2024-09-02", "2024-09-03", "2025-01-01", "2025-01-27",
    "2025-01-28", "2025-01-29", "2025-01-30", "2025-01-31", "2025-02-01", "2025-02-02", "2025-04-07", "2025-04-30",
    "2025-05-01", "2025-09-01", "2025-09-02"
]
holiday_days = pd.to_datetime(VN_HOLIDAY_DATES).normalize()
holiday_day_int = (holiday_days.astype("int64") // 86_400_000_000_000).to_numpy()

day_norm = df_final_all["TIME"].dt.normalize()
day_int = (day_norm.astype("int64") // 86_400_000_000_000).to_numpy()
df_final_all["is_vn_holiday"] = day_norm.isin(holiday_days).astype(float)
min_dist = np.min(np.abs(day_int[:, None] - holiday_day_int[None, :]), axis=1)
df_final_all["near_vn_holiday_3d"] = (min_dist <= 3).astype(float)


#  TỰ ĐỘNG HỌC GIỜ CAO ĐIỂM (CHƯA GÁN VÀO BASE_FEATURES)

df_final_all['Hour_float'] = df_final_all['TIME'].dt.hour + df_final_all['TIME'].dt.minute / 60.0

train_data_for_peak = df_final_all.iloc[:n_train]
top_peak_hours = train_data_for_peak.groupby('Hour_float')['P_224'].mean().nlargest(17).index.tolist()
df_final_all['is_peak'] = df_final_all['Hour_float'].isin(top_peak_hours).astype(float)

print(f" Tự động nhận diện 17 mốc 30-phút cao điểm nhất từ tập Train:")
print(f" -> Các mốc giờ (Thập phân): {sorted(top_peak_hours)}")

# 4. Gom nhóm
target_col = ['P_224']
substation_cols = [c for c in df_final_all.columns if c.startswith('P_') and not c.startswith('P224_') and c != 'P_224' and '_forecast' not in c and '_lag' not in c and 'Hour' not in c and 'Month' not in c and 'Day' not in c and 'split' not in c and 'TIME' not in c]

df_final_all['DayOfMonth'] = df_final_all['TIME'].dt.day
df_final_all['WeekOfYear'] = df_final_all['TIME'].dt.isocalendar().week.astype(int)
df_final_all['DayOfMonth_sin'] = np.sin(2 * np.pi * df_final_all['DayOfMonth'] / 31)
df_final_all['DayOfMonth_cos'] = np.cos(2 * np.pi * df_final_all['DayOfMonth'] / 31)
df_final_all['WeekOfYear_sin'] = np.sin(2 * np.pi * df_final_all['WeekOfYear'] / 52)
df_final_all['WeekOfYear_cos'] = np.cos(2 * np.pi * df_final_all['WeekOfYear'] / 52)

#  RÚT is_peak RA KHỎI ĐÂY ĐỂ BẢO TOÀN TRỌNG SỐ CHO MỐC 1, 2, 48
extra_time_cols = ['Hour_float', 'DayOfMonth_sin', 'DayOfMonth_cos', 'WeekOfYear_sin', 'WeekOfYear_cos']

 # Enhanced base_features với temporal patterns
base_features = forecast_weather_features + cyclical_cols + extra_time_cols + lag_features + temporal_pattern_features
topo_features = base_features + substation_cols

print(f"  Enhanced Features - Base: {len(base_features)} (: {len(base_features)-10}) | Topo: {len(topo_features)}")
print(f"    Thêm {len(temporal_pattern_features)} temporal features: {temporal_pattern_features}")
print(" Hoàn tất  Enhanced Feature Engineering.")

#  FINAL CHECK: Verify no NaN in new features
print(f"\n Final NaN check:")
for feature in temporal_pattern_features:
    nan_count = df_final_all[feature].isna().sum()
    print(f"   {feature}: {nan_count} NaN values")

if df_final_all[temporal_pattern_features].isna().sum().sum() > 0:
    print(" Still have NaN in temporal features, applying final fillna...")
    for feature in temporal_pattern_features:
        df_final_all[feature] = df_final_all[feature].fillna(0)
    print(" All temporal features cleaned")

 : Tạo Advanced Temporal Pattern Features...


/tmp/ipykernel_23/3709995793.py:59: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['weekly_same_hour'] = df[target_col].shift(7*48).fillna(method='bfill')  # Fill NaN


 Đã tạo 6 temporal pattern features: ['intraday_trend', 'weekly_same_hour', 'weekly_trend', 'load_gradient_1h', 'load_gradient_3h', 'load_acceleration']
Đã dọn dẹp xong NaN. Cắt bỏ 336 dòng mồ côi (chủ yếu ở đầu tập Train).
 Tự động nhận diện 17 mốc 30-phút cao điểm nhất từ tập Train:
 -> Các mốc giờ (Thập phân): [0.0, 0.5, 1.0, 1.5, 2.0, 18.0, 18.5, 19.0, 19.5, 20.0, 20.5, 21.0, 21.5, 22.0, 22.5, 23.0, 23.5]
  Enhanced Features - Base: 34 (: 24) | Topo: 38
    Thêm 6 temporal features: ['intraday_trend', 'weekly_same_hour', 'weekly_trend', 'load_gradient_1h', 'load_gradient_3h', 'load_acceleration']
 Hoàn tất  Enhanced Feature Engineering.

 Final NaN check:
   intraday_trend: 0 NaN values
   weekly_same_hour: 0 NaN values
   weekly_trend: 0 NaN values
   load_gradient_1h: 0 NaN values
   load_gradient_3h: 0 NaN values
   load_acceleration: 0 NaN values


In [3]:

# CẮT DATA & FIT SCALER (LỖI MẤT DỮ LIỆU)

WINDOW_SIZE = 48

# Lùi lại WINDOW_SIZE dòng để lấy bối cảnh cho tập Val và Test
df_train = df_final_all.iloc[:n_train].copy()
df_val   = df_final_all.iloc[n_train - WINDOW_SIZE : n_train + n_val].copy()
df_test  = df_final_all.iloc[n_train + n_val - WINDOW_SIZE : ].copy()

# Fit Scaler (Chỉ trên Train)
scaler_X_base = StandardScaler()
scaler_X_topo = StandardScaler()
scaler_y      = RobustScaler()

scaler_X_base.fit(df_train[base_features])
scaler_X_topo.fit(df_train[topo_features])
scaler_y.fit(df_train[target_col])

print("Fit xong Scaler trên tập Train.")

Fit xong Scaler trên tập Train.


In [4]:

#  HYBRID ARCHITECTURE (GRU VÀ TÁCH CHANNEL-INDEPENDENT AR)

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class Standard_KANLayer(nn.Module):
    def __init__(self, in_features, out_features, grid_size=5, spline_order=3):
        super().__init__()
        self.in_features, self.out_features, self.grid_size, self.spline_order = in_features, out_features, grid_size, spline_order
        self.base_activation = nn.GELU()
        self.base_weight   = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.spline_weight = nn.Parameter(torch.randn(out_features, in_features, grid_size + spline_order) * 0.1)
        self.register_buffer("grid", torch.linspace(-2.0, 2.0, grid_size + 1))

    def b_splines(self, x):
        x = x.unsqueeze(-1)
        h = (self.grid[1] - self.grid[0]).item()
        ext = torch.linspace(self.grid[0].item() - self.spline_order * h, self.grid[-1].item() + self.spline_order * h, self.grid_size + 1 + 2 * self.spline_order).to(x.device)
        view_shape = [1] * (x.dim() - 1) + [-1]
        ext = ext.view(*view_shape)
        bases = ((x >= ext[..., :-1]) & (x < ext[..., 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = ((x - ext[..., :-(k + 1)]) / (ext[..., k:-1] - ext[..., :-(k + 1)]) * bases[..., :-1] +
                     (ext[..., k + 1:] - x) / (ext[..., k + 1:] - ext[..., 1:-k]) * bases[..., 1:])
        return bases.contiguous()

    def update_grid(self, x, margin=0.01):
        with torch.no_grad():
            self.grid.copy_(torch.linspace(x.min().item() - margin, x.max().item() + margin, self.grid_size + 1).to(x.device))

    def forward(self, x):
        base_out = F.linear(self.base_activation(x), self.base_weight)
        if x.dim() == 2:
            spline_out = torch.einsum("big,oig->bo", self.b_splines(x), self.spline_weight)
        else:
            spline_out = torch.einsum("bsig,oig->bso", self.b_splines(x), self.spline_weight)
        return base_out + spline_out

class MS_KANLayer(nn.Module):
    def __init__(self, in_features, out_features, grid_size=5, spline_order=3):
        super().__init__()
        self.in_features, self.out_features = in_features, out_features
        self.spline_order = spline_order
        self.grid_size_c = grid_size
        self.grid_size_f = grid_size * 2
        self.base_activation = nn.GELU()
        self.base_weight = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.spline_weight_c = nn.Parameter(torch.randn(out_features, in_features, self.grid_size_c + spline_order) * 0.1)
        self.spline_weight_f = nn.Parameter(torch.randn(out_features, in_features, self.grid_size_f + spline_order) * 0.05)
        self.register_buffer("grid_c", torch.linspace(-2.0, 2.0, self.grid_size_c + 1))
        self.register_buffer("grid_f", torch.linspace(-2.0, 2.0, self.grid_size_f + 1))

    def b_splines(self, x, grid, grid_size):
        x = x.unsqueeze(-1)
        h = (grid[1] - grid[0]).item()
        ext = torch.linspace(grid[0].item() - self.spline_order * h, grid[-1].item() + self.spline_order * h, grid_size + 1 + 2 * self.spline_order).to(x.device)
        view_shape = [1] * (x.dim() - 1) + [-1]
        ext = ext.view(*view_shape)
        bases = ((x >= ext[..., :-1]) & (x < ext[..., 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = ((x - ext[..., :-(k + 1)]) / (ext[..., k:-1] - ext[..., :-(k + 1)]) * bases[..., :-1] +
                     (ext[..., k + 1:] - x) / (ext[..., k + 1:] - ext[..., 1:-k]) * bases[..., 1:])
        return bases.contiguous()

    def update_grid(self, x, margin=0.01):
        with torch.no_grad():
            self.grid_c.copy_(torch.linspace(x.min().item() - margin, x.max().item() + margin, self.grid_size_c + 1).to(x.device))
            self.grid_f.copy_(torch.linspace(x.min().item() - margin, x.max().item() + margin, self.grid_size_f + 1).to(x.device))

    def forward(self, x):
        base_out = F.linear(self.base_activation(x), self.base_weight)
        spline_c = self.b_splines(x, self.grid_c, self.grid_size_c)
        spline_f = self.b_splines(x, self.grid_f, self.grid_size_f)
        if x.dim() == 2:
            out_c = torch.einsum("big,oig->bo", spline_c, self.spline_weight_c)
            out_f = torch.einsum("big,oig->bo", spline_f, self.spline_weight_f)
        else:
            out_c = torch.einsum("bsig,oig->bso", spline_c, self.spline_weight_c)
            out_f = torch.einsum("bsig,oig->bso", spline_f, self.spline_weight_f)
        return base_out + out_c + out_f

    def regularization_loss(self, lamb_l1=0.01):
        return lamb_l1 * (self.spline_weight_c.abs().mean() + 2.0 * self.spline_weight_f.abs().mean())

class Hybrid_KANLayer(nn.Module):
    def __init__(self, in_features, out_features, grid_size=5, spline_order=3, hybrid_ratio=0.5):
        super().__init__()
        self.in_features, self.out_features = in_features, out_features
        self.hybrid_ratio = hybrid_ratio
        self.standard_kan = Standard_KANLayer(in_features, out_features, grid_size, spline_order)
        self.ms_kan = MS_KANLayer(in_features, out_features, grid_size, spline_order)
        self.weight_gate = nn.Sequential(nn.Linear(in_features, 32), nn.GELU(), nn.Linear(32, 2), nn.Softmax(dim=-1))
        self.cross_attention = nn.MultiheadAttention(out_features, num_heads=4, batch_first=True)
        self.norm = nn.LayerNorm(out_features)

    def update_grid(self, x, margin=0.01):
        self.standard_kan.update_grid(x, margin)
        self.ms_kan.update_grid(x, margin)

    def forward(self, x):
        std_out = self.standard_kan(x)
        ms_out = self.ms_kan(x)
        if x.dim() == 2:
            weights = self.weight_gate(x.mean(dim=0, keepdim=True)).expand(x.size(0), -1)
        else:
            weights = self.weight_gate(x.mean(dim=1))

        w_std, w_ms = weights[:, 0:1], weights[:, 1:2]
        if x.dim() == 3: w_std, w_ms = w_std.unsqueeze(1), w_ms.unsqueeze(1)

        if x.dim() == 2:
            std_out_seq, ms_out_seq = std_out.unsqueeze(1), ms_out.unsqueeze(1)
            attn_out, _ = self.cross_attention(std_out_seq, ms_out_seq, ms_out_seq)
            attn_out = attn_out.squeeze(1)
            hybrid_out = w_std * std_out + w_ms * ms_out + 0.1 * attn_out
        else:
            attn_out, _ = self.cross_attention(std_out, ms_out, ms_out)
            hybrid_out = w_std * std_out + w_ms * ms_out + 0.1 * attn_out
        return self.norm(hybrid_out)

    def regularization_loss(self, lamb_l1=0.01):
        return (self.standard_kan.spline_weight.abs().mean() + self.ms_kan.regularization_loss(1.0)) * lamb_l1

class Enhanced_SparseTopologyEncoder(nn.Module):
    def __init__(self, seq_len, num_features, topo_dim=32, num_layers=2, dropout=0.1, topk=4):
        super().__init__()
        self.num_features, self.topo_dim, self.topk = num_features, topo_dim, topk
        self.conv_short = nn.Conv1d(num_features, num_features, kernel_size=3, padding=1, groups=num_features)
        self.conv_medium = nn.Conv1d(num_features, num_features, kernel_size=5, padding=2, groups=num_features)
        self.conv_long = nn.Conv1d(num_features, num_features, kernel_size=7, padding=3, groups=num_features)
        self.temporal_fusion = nn.Linear(num_features * 3, num_features)
        self.temporal_proj = nn.Linear(seq_len, topo_dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.msg_mlps = nn.ModuleList([nn.Sequential(nn.Linear(topo_dim, topo_dim * 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(topo_dim * 2, topo_dim)) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(topo_dim) for _ in range(num_layers)])
        self.gates = nn.ModuleList([nn.Sequential(nn.Linear(topo_dim, topo_dim), nn.Sigmoid()) for _ in range(num_layers)])
        self.pool_proj = nn.Linear(topo_dim * 3, topo_dim)
        self.attention_pool = nn.MultiheadAttention(topo_dim, num_heads=4, batch_first=True)
        self.node_emb = nn.Parameter(torch.randn(num_features, topo_dim) * 0.1)

    def build_adj(self, device):
        sim = torch.matmul(self.node_emb, self.node_emb.t()) / math.sqrt(self.topo_dim)
        A = torch.relu(sim) + torch.eye(self.num_features, device=device)
        if self.topk < self.num_features:
            vals, idx = torch.topk(A, k=max(1, self.topk), dim=-1)
            mask = torch.zeros_like(A).scatter_(1, idx, 1.0)
            A = A * mask
        return A / A.sum(dim=-1, keepdim=True).clamp_min(1e-6)

    def forward(self, x):
        x_t = x.transpose(1, 2)
        multi_scale = torch.cat([self.conv_short(x_t), self.conv_medium(x_t), self.conv_long(x_t)], dim=1).transpose(1, 2)
        fused = self.temporal_fusion(multi_scale)
        h = self.act(self.temporal_proj(fused.transpose(1, 2)))
        A_b = self.build_adj(x.device).unsqueeze(0).expand(h.size(0), -1, -1)
        global_ctx = h.mean(dim=1, keepdim=True)
        for msg_mlp, gate, norm in zip(self.msg_mlps, self.gates, self.norms):
            h = norm(h + self.dropout(self.act(msg_mlp(torch.bmm(A_b, h) + global_ctx) * gate(h))))
        h_attn, _ = self.attention_pool(h, h, h)
        return self.pool_proj(torch.cat([h.mean(dim=1), h.max(dim=1).values, h_attn.mean(dim=1)], dim=-1))


# ------------------------------------------------------------------------------
# 5A. TOPO_KAN (STANDARD) - GRU & AR CHANNEL-INDEPENDENT
# ------------------------------------------------------------------------------
class Topo_KAN_Standard(nn.Module):
    def __init__(self, seq_len, num_features, output_dim, hidden_dim=64, topo_dim=32, fusion_dim=128, dropout=0.1, grid_size=5, lag_indices=None):
        super().__init__()
        self.lag_indices = lag_indices if lag_indices is not None else list(range(num_features))

        self.gru = nn.GRU(input_size=len(self.lag_indices), hidden_size=hidden_dim, batch_first=True)
        self.topo_encoder = Enhanced_SparseTopologyEncoder(seq_len, num_features, topo_dim=topo_dim, dropout=dropout)

        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim + topo_dim, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, fusion_dim),
            nn.GELU()
        )

        self.ar_heads = nn.ModuleList([nn.Linear(seq_len * len(self.lag_indices), 1) for _ in range(output_dim)])

        self.kan_layer1 = Hybrid_KANLayer(fusion_dim, hidden_dim, grid_size=grid_size, hybrid_ratio=0.3)
        self.kan_layer2 = Standard_KANLayer(hidden_dim, output_dim, grid_size=grid_size)
        self.topo_to_hidden = nn.Linear(topo_dim, hidden_dim)
        self.topo_to_out_gate = nn.Sequential(nn.Linear(topo_dim, output_dim), nn.Sigmoid())

    def forward(self, x):
        topo_vec = self.topo_encoder(x)
        x_lags = x[:, :, self.lag_indices]

        _, h_n = self.gru(x_lags)
        gru_out = h_n[-1]

        fused = self.fusion(torch.cat([gru_out, topo_vec], dim=1))

        x_lags_flat = x_lags.reshape(x.size(0), -1)
        ar_out = torch.cat([head(x_lags_flat) for head in self.ar_heads], dim=1)

        hidden = self.kan_layer1(fused) + self.topo_to_hidden(topo_vec)
        return ar_out + self.topo_to_out_gate(topo_vec) * self.kan_layer2(hidden)

    def regularization_loss(self, lamb_l1=0.01):
        return lamb_l1 * (self.kan_layer1.regularization_loss(1.0) + self.kan_layer2.spline_weight.abs().mean())

# ------------------------------------------------------------------------------
# 5B. TOPO_KAN (MS) - GRU & AR CHANNEL-INDEPENDENT
# ------------------------------------------------------------------------------
class Topo_KAN_MS(nn.Module):
    def __init__(self, seq_len, num_features, output_dim, hidden_dim=64, topo_dim=32, fusion_dim=128, dropout=0.1, grid_size=5, lag_indices=None):
        super().__init__()
        self.lag_indices = lag_indices if lag_indices is not None else list(range(num_features))

        self.gru = nn.GRU(input_size=len(self.lag_indices), hidden_size=hidden_dim, batch_first=True)
        self.topo_encoder = Enhanced_SparseTopologyEncoder(seq_len, num_features, topo_dim=topo_dim, dropout=dropout)

        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim + topo_dim, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.ar_heads = nn.ModuleList([nn.Linear(seq_len * len(self.lag_indices), 1) for _ in range(output_dim)])
        self.kan_layer1 = Hybrid_KANLayer(fusion_dim, hidden_dim, grid_size=grid_size, hybrid_ratio=0.7)
        self.kan_layer2 = MS_KANLayer(hidden_dim, output_dim, grid_size=grid_size)
        self.topo_to_hidden = nn.Linear(topo_dim, hidden_dim)
        self.topo_to_out_gate = nn.Sequential(nn.Linear(topo_dim, output_dim), nn.Sigmoid())

    def forward(self, x):
        topo_vec = self.topo_encoder(x)
        x_lags = x[:, :, self.lag_indices]

        _, h_n = self.gru(x_lags)
        gru_out = h_n[-1]

        fused = self.fusion(torch.cat([gru_out, topo_vec], dim=1))

        x_lags_flat = x_lags.reshape(x.size(0), -1)
        ar_out = torch.cat([head(x_lags_flat) for head in self.ar_heads], dim=1)

        hidden = self.kan_layer1(fused) + self.topo_to_hidden(topo_vec)
        return ar_out + self.topo_to_out_gate(topo_vec) * self.kan_layer2(hidden)

    def regularization_loss(self, lamb_l1=0.01):
        return lamb_l1 * (self.kan_layer1.regularization_loss(1.0) + self.kan_layer2.regularization_loss(1.0))

print(" Đã nạp Kiến trúc  Fix chuẩn: GRU, Tách AR Head.")

 Đã nạp Kiến trúc  Fix chuẩn: GRU, Tách AR Head.


In [5]:
# ==============================================================================
# CELL 5: ULTIMATE TRAINING PIPELINE (COMPOSITE LOSS + ENSEMBLE + COMPLEXITY)
# ==============================================================================

from sklearn.metrics import mean_squared_error
import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import time # [NEW] Bổ sung thư viện đo thời gian

#  HÀM LOSS "THẬP CẨM" (COMPOSITE LOSS)
def composite_forecast_loss(y_pred, y_true, penalty_under=2.0, penalty_over=0.5):
    error = y_true - y_pred

    # 1. Huber Loss Bất đối xứng (Chống nổ Gradient)
    base_huber = F.huber_loss(y_pred, y_true, reduction='none', delta=0.1)
    asym_huber = torch.where(error > 0, penalty_under * base_huber, penalty_over * base_huber)
    mean_asym_loss = asym_huber.mean()

    # 2. WAPE-like Term (Ép trực tiếp điểm WAPE)
    eps = 1e-6
    abs_error = torch.abs(error)
    wape_loss = torch.mean(abs_error) / (torch.mean(torch.abs(y_true)) + eps)

    # 3. Peak Weighting (Bắt đỉnh: Phạt nặng ở vùng tải cao)
    peak_weight = torch.clamp(y_true, min=0.0)
    peak_loss = torch.mean(peak_weight * abs_error)

    # 4. Smoothness & Curvature (Đạo hàm bậc 1 & bậc 2)
    if y_pred.size(1) > 2:
        dy_pred = y_pred[:, 1:] - y_pred[:, :-1]
        dy_true = y_true[:, 1:] - y_true[:, :-1]
        smooth_loss = F.mse_loss(dy_pred, dy_true)

        ddy_pred = dy_pred[:, 1:] - dy_pred[:, :-1]
        ddy_true = dy_true[:, 1:] - dy_true[:, :-1]
        curve_loss = F.mse_loss(ddy_pred, ddy_true)
    else:
        smooth_loss = torch.tensor(0.0, device=y_pred.device)
        curve_loss = torch.tensor(0.0, device=y_pred.device)

    # Gộp toàn bộ siêu Loss
    w_huber, w_wape, w_peak, w_smooth, w_curve = 1.0, 0.4, 0.3, 0.1, 0.1

    total_loss = (w_huber * mean_asym_loss +
                  w_wape * wape_loss +
                  w_peak * peak_loss +
                  w_smooth * smooth_loss +
                  w_curve * curve_loss)
    return total_loss

def calibrate_kan_layers(model, calib_x):
    hooks = []
    def hook_fn(module, input, output):
        if hasattr(module, 'update_grid'): module.update_grid(input[0])
    for name, module in model.named_modules():
        if hasattr(module, 'update_grid'): hooks.append(module.register_forward_hook(hook_fn))
    with torch.no_grad(): model(calib_x)
    for h in hooks: h.remove()

def calculate_metrics(act, pre):
    from sklearn.metrics import mean_absolute_error, r2_score
    eps = 1e-10
    mae = mean_absolute_error(act, pre)
    rmse = np.sqrt(mean_squared_error(act, pre))
    r2 = r2_score(act, pre)
    wape = np.sum(np.abs(act - pre)) / (np.sum(np.abs(act)) + eps) * 100
    return mae, rmse, wape, r2

def train_single_model_v68(model_obj, name, loader, val_loader, device,
                          forecast_steps, pen_under, pen_over, opt_type="Adam", sched_type="Reduce",
                          reg_lambda=0.005, epochs=100):

    model_obj.eval()
    with torch.no_grad():
        calib_x, n = [], 0
        for bx, _ in loader:
            calib_x.append(bx.to(device)); n += bx.size(0)
            if n >= 512: break
        calib_x = torch.cat(calib_x, dim=0)[:512]
        calibrate_kan_layers(model_obj, calib_x)

    # LR
    if forecast_steps in [1, 2]:
        base_lr = 2e-3; weight_decay = 5e-6; patience = 5; factor = 0.7; grad_clip = 0.8
    elif forecast_steps in [6]:
        base_lr = 1.5e-3; weight_decay = 1e-5; patience = 4; factor = 0.6; grad_clip = 1.0
    else:
        base_lr = 8e-4; weight_decay = 2e-5; patience = 6; factor = 0.5; grad_clip = 1.2
        reg_lambda = reg_lambda * 0.7

    if opt_type == "AdamW":
        optimizer = optim.AdamW(model_obj.parameters(), lr=base_lr, weight_decay=weight_decay, betas=(0.9, 0.999), eps=1e-8)
    else:
        optimizer = optim.Adam(model_obj.parameters(), lr=base_lr, weight_decay=weight_decay, betas=(0.9, 0.999), eps=1e-8)

    if sched_type == "Cosine":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=base_lr*0.01)
    elif sched_type == "CosineWarmup":
        warmup_epochs = min(10, epochs // 10)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs-warmup_epochs, eta_min=base_lr*0.01)
        warmup_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    else:
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=factor, patience=patience, min_lr=base_lr*0.001)

    best_loss, counter, ckpt = float('inf'), 0, f"best_{name}.pth"

    for epoch in range(epochs):
        model_obj.train()
        train_losses = []

        if sched_type == "CosineWarmup" and epoch < warmup_epochs: current_scheduler = warmup_scheduler
        else: current_scheduler = scheduler

        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            pred = model_obj(bx)

            loss = composite_forecast_loss(pred, by, pen_under, pen_over) + model_obj.regularization_loss(reg_lambda)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_obj.parameters(), max_norm=grad_clip)
            optimizer.step()
            train_losses.append(loss.item())

        if sched_type in ["Cosine", "CosineWarmup"]: current_scheduler.step()

        model_obj.eval()
        val_l = 0.0
        with torch.no_grad():
            for bx_v, by_v in val_loader:
                bx_v, by_v = bx_v.to(device), by_v.to(device)
                pred_v = model_obj(bx_v)
                val_l += composite_forecast_loss(pred_v, by_v, pen_under, pen_over).item()
        val_l /= len(val_loader)

        if sched_type == "Reduce": scheduler.step(val_l)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"      Epoch {epoch+1:03d}/{epochs} | Train Loss: {np.mean(train_losses):.6f} | Val Loss: {val_l:.6f} | LR: {current_lr:.2e}")

        if val_l < best_loss:
            best_loss, counter = val_l, 0
            torch.save(model_obj.state_dict(), ckpt)
        else:
            counter += 1
            early_stop_patience = 20 if forecast_steps >= 12 else 15
            if counter >= early_stop_patience:
                print(f"      => Early stopping tại epoch {epoch+1}. Best Val Loss: {best_loss:.6f}")
                break

    model_obj.load_state_dict(torch.load(ckpt, map_location=device))
    return model_obj

def run_pipeline_for_horizon_v68(forecast_steps, model_class, use_topo=False, epochs=120):
    mode_name = "TOPO_ENSEMBLE" if use_topo else "BASE_ENSEMBLE"
    print(f"\n{'='*75}")
    print(f" ULTIMATE TRAINING: {model_class.__name__} | MỐC: {forecast_steps} ĐIỂM")
    print(f"  Architecture:  Hybrid KAN + GRU + AR-Independent")
    print(f"  Loss Function:  Composite Loss (WAPE + Smoothness + Peak)")
    print(f"{'='*75}")

    WINDOW_SIZE = 48
    active_features = topo_features.copy() if use_topo else base_features.copy()

    if forecast_steps == 2 and 'P224_lag1' not in active_features:
        idx = active_features.index('P224_lag2')
        active_features.insert(idx, 'P224_lag1')
    if forecast_steps == 12 and "is_vn_holiday" not in active_features:
        active_features.extend(["is_vn_holiday", "near_vn_holiday_3d"])
    if forecast_steps in [6, 12, 24] and "is_peak" not in active_features:
        idx_peak = active_features.index('WeekOfYear_cos') + 1
        active_features.insert(idx_peak, "is_peak")

    lag_indices = [i for i, f in enumerate(active_features) if 'P224_' in f or f == 'P_224']

    from sklearn.preprocessing import StandardScaler
    active_scaler_X = StandardScaler().fit(df_train[active_features])

    def create_custom_seq(df, s_x, s_y):
        X_raw = s_x.transform(df[active_features])
        y_raw = s_y.transform(df[target_col])
        X, y = [], []
        limit = len(df) - WINDOW_SIZE - forecast_steps + 1
        for i in range(limit):
            X.append(X_raw[i : i + WINDOW_SIZE])
            y.append(y_raw[i + WINDOW_SIZE : i + WINDOW_SIZE + forecast_steps].flatten())
        return torch.from_numpy(np.array(X, dtype=np.float32)), torch.from_numpy(np.array(y, dtype=np.float32))

    X_tr, y_tr = create_custom_seq(df_train, active_scaler_X, scaler_y)
    X_va, y_va = create_custom_seq(df_val,   active_scaler_X, scaler_y)
    X_te, y_te = create_custom_seq(df_test,  active_scaler_X, scaler_y)

    if forecast_steps in [1, 2]: batch_size = 128
    elif forecast_steps in [6]: batch_size = 96
    else: batch_size = 64

    loader_args = {'batch_size': batch_size, 'shuffle': False}
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), **loader_args)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), **loader_args)

    if forecast_steps >= 12:
        base_config = {"hd": 64, "gs": 5, "opt": "AdamW", "sch": "CosineWarmup", "reg": 0.003}
        p_under, p_over = 1.5, 0.5
        print("  [HORIZON-AWARE] Dài Hạn -> Loss Ratio (Under/Over) = 1.5 / 0.5")
    else:
        base_config = {"hd": 64, "gs": 5, "opt": "Adam", "sch": "Reduce", "reg": 0.005}
        p_under, p_over = 2.0, 0.5
        print("  [HORIZON-AWARE] Ngắn Hạn -> Loss Ratio (Under/Over) = 2.0 / 0.5")

    ensemble_seeds = [42, 123, 456]
    all_preds = []
    
    # [NEW] Các biến theo dõi Complexity
    total_train_time = 0.0
    total_inf_time = 0.0
    total_test_samples = len(X_te) * len(ensemble_seeds)

    # [NEW] Đo Parameter Count (Đếm trên 1 model mẫu vì kiến trúc y hệt nhau)
    dummy_model = model_class(WINDOW_SIZE, len(active_features), forecast_steps,
                              hidden_dim=base_config["hd"], grid_size=base_config["gs"],
                              lag_indices=lag_indices)
    total_params = sum(p.numel() for p in dummy_model.parameters() if p.requires_grad)

    for i, seed in enumerate(ensemble_seeds):
        print(f" -> Đang luyện Model {i+1}/3 (Seed: {seed})...")
        torch.manual_seed(seed)
        np.random.seed(seed)

        model_obj = model_class(WINDOW_SIZE, len(active_features), forecast_steps,
                               hidden_dim=base_config["hd"], grid_size=base_config["gs"],
                               lag_indices=lag_indices).to(device)

        m_name = f"{model_class.__name__}_{forecast_steps}P_M{i+1}"

        # [NEW] Đo Training Time
        start_train = time.time()
        trained_model = train_single_model_v68(model_obj, m_name, train_loader, val_loader, device,
                                              forecast_steps, p_under, p_over,
                                              base_config["opt"], base_config["sch"], base_config["reg"], epochs)
        total_train_time += (time.time() - start_train)

        trained_model.eval()
        preds = []
        
        # [NEW] Đo Inference Time
        start_inf = time.time()
        with torch.no_grad():
            for bx, _ in test_loader:
                preds.append(trained_model(bx.to(device)).cpu().numpy())
        total_inf_time += (time.time() - start_inf)

        current_preds = np.concatenate(preds)
        all_preds.append(current_preds)

        flat_actuals = y_te.numpy().reshape(-1, 1)
        flat_preds_single = current_preds.reshape(-1, 1)
        inv_act_single = scaler_y.inverse_transform(flat_actuals).reshape(-1, forecast_steps)
        inv_pre_single = scaler_y.inverse_transform(flat_preds_single).reshape(-1, forecast_steps)

        act_non_overlap_single = inv_act_single[::forecast_steps, :].flatten()
        pre_non_overlap_single = inv_pre_single[::forecast_steps, :].flatten()

        mae_s, rmse_s, wape_s, r2_s = calculate_metrics(act_non_overlap_single, pre_non_overlap_single)
        print(f"    => [KẾT QUẢ ĐỘC LẬP MODEL {i+1}/3]: WAPE: {wape_s:.2f}% | MAE: {mae_s:.2f} | R2: {r2_s:.4f}\n")

    avg_preds = np.mean(all_preds, axis=0)
    flat_preds = avg_preds.reshape(-1, 1)

    inv_act_flat = scaler_y.inverse_transform(flat_actuals)
    inv_pre_flat = scaler_y.inverse_transform(flat_preds)

    inv_act = inv_act_flat.reshape(-1, forecast_steps)
    inv_pre = inv_pre_flat.reshape(-1, forecast_steps)

    act_non_overlap = inv_act[::forecast_steps, :].flatten()
    pre_non_overlap = inv_pre[::forecast_steps, :].flatten()

    mae, rmse, wape, r2 = calculate_metrics(act_non_overlap, pre_non_overlap)
    
    # [NEW] Tính toán ms / sample
    avg_inf_ms = (total_inf_time / total_test_samples) * 1000
    
    print(f" [KẾT QUẢ ENSEMBLE CHUNG CUỘC]: WAPE: {wape:.2f}% | MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.4f} | Params: {total_params/1000:.1f}K | Train: {total_train_time:.1f}s")

    return {
        "Mô hình": f"{model_class.__name__} (Ultimate)", 
        "Mốc (Điểm)": forecast_steps, 
        "MAE": mae, 
        "RMSE": rmse, 
        "WAPE (%)": wape, 
        "R2": r2,
        "Params (K)": round(total_params / 1000, 1),   # [NEW] Thêm Cột Params
        "Train_Time (s)": round(total_train_time, 1),  # [NEW] Thêm Cột Train Time
        "Inf_Time (ms)": round(avg_inf_ms, 4)          # [NEW] Thêm Cột Inference Time
    }

print(" Đã nạp Pipeline Ultimate: Composite Loss + Ensemble 3 Models + Complexity Tracker.")

 Đã nạp Pipeline Ultimate: Composite Loss + Ensemble 3 Models + Complexity Tracker.


In [6]:
# ==============================================================================
# CELL 6: VÒNG LẶP CHẠY SO SÁNH (ĐÃ THÊM COMPLEXITY ANALYSIS)
# ==============================================================================

# Test horizons like original kanb-kant với enhanced training
horizons = [
    1, 2,
    6, 12,
    24, 48
]

results = []

print("  TEST: Enhanced Training với Horizon-Specific Optimization")
for forecast_steps in horizons:
    try:
        # Determine model type based on horizon
        if forecast_steps in [1, 2, 6]:
            print(f"\nMốc NGẮN ({forecast_steps}P) -> Kích hoạt MS-KAN với Enhanced Training")
            model_class = Topo_KAN_MS
        else:
            print(f"\nMốc DÀI ({forecast_steps}P) -> Kích hoạt Standard Topo_KAN với Enhanced Training")
            model_class = Topo_KAN_Standard

        # Run enhanced pipeline
        result = run_pipeline_for_horizon_v68(forecast_steps, model_class, use_topo=True, epochs=120)
        results.append(result)

    except Exception as e:
        print(f" Lỗi tại {forecast_steps}P: {e}")
        continue

# Display results table
if results:
    # [NEW] Mở rộng bảng pandas để không bị rớt dòng khi thêm 3 cột mới
    pd.set_option('display.max_rows', None)
    pd.set_option('display.width', 1500)
    pd.set_option('display.max_columns', None)
    
    print(f"\n{'='*120}")
    print("  KẾT QUẢ CUỐI CÙNG - ENHANCED TRAINING KÈM ĐÁNH GIÁ ĐỘ PHỨC TẠP")
    print(f"{'='*120}")
    df_results = pd.DataFrame(results)
    print(df_results.to_string(index=False))

    # Summary
    avg_wape = df_results['WAPE (%)'].mean()
    print(f"\n WAPE Trung bình : {avg_wape:.2f}%")

    # Compare with temporal baseline
    if len(results) > 0:
        wape_1p = df_results[df_results['Mốc (Điểm)'] == 1]['WAPE (%)'].iloc[0]
        print(f"  1P WAPE: {wape_1p:.2f}%")
        print(f"  Strategy: Same architecture + Better training process")

        # Show improvements by horizon
        print(f"\n  Horizon Analysis:")
        for _, row in df_results.iterrows():
            horizon = row['Mốc (Điểm)']
            wape = row['WAPE (%)']
            if horizon in [1, 2, 6]:
                print(f"   {horizon}P ({horizon*0.5}H): {wape:.2f}% (Short-term với enhanced batch size)")
            else:
                print(f"   {horizon}P ({horizon*0.5}H): {wape:.2f}% (Long-term với reduced penalty + AdamW)")
else:
    print(" Không có kết quả nào được tạo ra")

print("\n Hoàn tất Enhanced Training Test!")
print("  Focus: Better hyperparameters, not architecture changes")

  TEST: Enhanced Training với Horizon-Specific Optimization

Mốc NGẮN (1P) -> Kích hoạt MS-KAN với Enhanced Training

 ULTIMATE TRAINING: Topo_KAN_MS | MỐC: 1 ĐIỂM
  Architecture:  Hybrid KAN + GRU + AR-Independent
  Loss Function:  Composite Loss (WAPE + Smoothness + Peak)
  [HORIZON-AWARE] Ngắn Hạn -> Loss Ratio (Under/Over) = 2.0 / 0.5
 -> Đang luyện Model 1/3 (Seed: 42)...
      Epoch 001/120 | Train Loss: 0.129650 | Val Loss: 0.121187 | LR: 2.00e-03
      Epoch 010/120 | Train Loss: 0.090714 | Val Loss: 0.108540 | LR: 2.00e-03
      Epoch 020/120 | Train Loss: 0.082773 | Val Loss: 0.081511 | LR: 1.40e-03
      Epoch 030/120 | Train Loss: 0.077145 | Val Loss: 0.078839 | LR: 9.80e-04
      Epoch 040/120 | Train Loss: 0.073489 | Val Loss: 0.079327 | LR: 6.86e-04
      Epoch 050/120 | Train Loss: 0.069372 | Val Loss: 0.080108 | LR: 3.36e-04
      Epoch 060/120 | Train Loss: 0.067460 | Val Loss: 0.073192 | LR: 2.35e-04
      Epoch 070/120 | Train Loss: 0.066383 | Val Loss: 0.073739 | L